In [8]:
import duckdb
df = duckdb.query("""
    SELECT 
        sourceTaxonName, 
        sourceTaxonRank,
        targetTaxonName, 
        targetTaxonRank,
        referenceCitation,
        interactionTypeName
    FROM 'interactions.csv.gz'
    WHERE referenceCitation LIKE '%www.inaturalist.org/observations%'
""").to_df()


In [9]:
df["referenceCitation"] = df["referenceCitation"].str.extract(
    r"(https?://[^\s]+)"
)

In [10]:
df = df.dropna(subset=["sourceTaxonRank"])

In [11]:
GBIF_CORE_RANKS = [
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
]

df = df[
    df["sourceTaxonRank"].isin(GBIF_CORE_RANKS)
    & df["targetTaxonRank"].isin(GBIF_CORE_RANKS)
]
df

,sourceTaxonName,sourceTaxonRank,targetTaxonName,targetTaxonRank,referenceCitation,interactionTypeName
7,Felis,genus,Sciurus niger,species,https://www.inaturalist.org/observations/74666844,kills
16,Turdus migratorius,species,Magicicada,genus,https://www.inaturalist.org/observations/82003741,eats
17,Turdus migratorius,species,Magicicada,genus,https://www.inaturalist.org/observations/82003741,preysOn
19,Vanessa itea,species,Escallonia laevis,species,https://www.inaturalist.org/observations/11465631,interactsWith
20,Ahaetulla borealis,species,Sauria,genus,https://www.inaturalist.org/observations/82005152,eats
...,...,...,...,...,...,...
1271250,Hydrocharis morsus-ranae,species,Nymphaea odorata,species,https://www.inaturalist.org/observations/55633954,interactsWith
1271251,Hydrocharis morsus-ranae,species,Lemna,genus,https://www.inaturalist.org/observations/55633954,interactsWith
1271252,Hydrocharis morsus-ranae,species,Typha,genus,https://www.inaturalist.org/observations/55633954,interactsWith
1271253,Hydrocharis morsus-ranae,species,Ceratophyllum demersum,species,https://www.inaturalist.org/observations/55633954,interactsWith


In [12]:
import duckdb
import zipfile
import tempfile
import shutil

with zipfile.ZipFile("backbone.zip") as z:
    taxon_path = [name for name in z.namelist() if name.endswith("Taxon.tsv")][0]

    with z.open(taxon_path) as src, tempfile.NamedTemporaryFile(suffix=".tsv") as tmp:
        shutil.copyfileobj(src, tmp)
        tmp.flush()

        df_taxon = duckdb.sql(f"""
            SELECT *
            FROM read_csv(
                '{tmp.name}',
                delim='\t',
                header=True,
                ignore_errors=true
            )
        """).to_df()

df_taxon.head()

,taxonID,datasetID,parentNameUsageID,acceptedNameUsageID,originalNameUsageID,scientificName,scientificNameAuthorship,canonicalName,genericName,specificEpithet,...,namePublishedIn,taxonomicStatus,nomenclaturalStatus,taxonRemarks,kingdom,phylum,class,order,family,genus
0,11838827,61a5f178-b5fb-4484-b6d8-9b129739e59d,5,None,None,SH1076374.09FU,None,None,None,None,...,None,accepted,None,None,Fungi,None,None,None,None,None
1,11857967,61a5f178-b5fb-4484-b6d8-9b129739e59d,5,None,None,SH1048761.09FU,None,None,None,None,...,None,accepted,None,None,Fungi,None,None,None,None,None
2,11693236,61a5f178-b5fb-4484-b6d8-9b129739e59d,5,None,None,SH1317510.09FU,None,None,None,None,...,None,accepted,None,None,Fungi,None,None,None,None,None
3,11525917,61a5f178-b5fb-4484-b6d8-9b129739e59d,5,None,None,SH1111669.09FU,None,None,None,None,...,None,accepted,None,None,Fungi,None,None,None,None,None
4,11646423,61a5f178-b5fb-4484-b6d8-9b129739e59d,5,None,None,SH0969972.09FU,None,None,None,None,...,None,accepted,None,None,Fungi,None,None,None,None,None


In [13]:
df_taxon = df_taxon.dropna(subset=["canonicalName"])
df_taxon = df_taxon[df_taxon["taxonomicStatus"] == "accepted"]
df_taxon = df_taxon[
    ["taxonID", "canonicalName", "kingdom", "phylum", "class", "order", "family", "genus"]
]

In [14]:
taxon_map = df_taxon[[
    "canonicalName", "kingdom", "phylum", "class", "order", "family", "genus", "taxonID"
]]

df = df.merge(taxon_map, how="left", left_on="sourceTaxonName", right_on="canonicalName")

df = df.rename(columns={
    "kingdom": "sourceTaxonKingdomName",
    "phylum": "sourceTaxonPhylumName",
    "class": "sourceTaxonClassName",
    "order": "sourceTaxonOrderName",
    "family": "sourceTaxonFamilyName",
    "genus": "sourceTaxonGenusName",
}).drop(columns=["canonicalName"])

df = df.merge(taxon_map, how="left", left_on="targetTaxonName", right_on="canonicalName")

df = df.rename(columns={
    "kingdom": "targetTaxonKingdomName",
    "phylum": "targetTaxonPhylumName",
    "class": "targetTaxonClassName",
    "order": "targetTaxonOrderName",
    "family": "targetTaxonFamilyName",
    "genus": "targetTaxonGenusName",
}).drop(columns=["canonicalName"])
df

,sourceTaxonName,sourceTaxonRank,targetTaxonName,targetTaxonRank,referenceCitation,interactionTypeName,sourceTaxonKingdomName,sourceTaxonPhylumName,sourceTaxonClassName,sourceTaxonOrderName,sourceTaxonFamilyName,sourceTaxonGenusName,taxonID_x,targetTaxonKingdomName,targetTaxonPhylumName,targetTaxonClassName,targetTaxonOrderName,targetTaxonFamilyName,targetTaxonGenusName,taxonID_y
0,Felis,genus,Sciurus niger,species,https://www.inaturalist.org/observations/74666844,kills,Animalia,Chordata,Mammalia,Carnivora,Felidae,Felis,2435022.0,Animalia,Chordata,Mammalia,Rodentia,Sciuridae,Sciurus,5219683.0
1,Turdus migratorius,species,Magicicada,genus,https://www.inaturalist.org/observations/82003741,eats,Animalia,Chordata,Aves,Passeriformes,Turdidae,Turdus,9510564.0,Animalia,Arthropoda,Insecta,Hemiptera,Cicadidae,Magicicada,4778821.0
2,Turdus migratorius,species,Magicicada,genus,https://www.inaturalist.org/observations/82003741,preysOn,Animalia,Chordata,Aves,Passeriformes,Turdidae,Turdus,9510564.0,Animalia,Arthropoda,Insecta,Hemiptera,Cicadidae,Magicicada,4778821.0
3,Vanessa itea,species,Escallonia laevis,species,https://www.inaturalist.org/observations/11465631,interactsWith,Animalia,Arthropoda,Insecta,Lepidoptera,Nymphalidae,Vanessa,5806175.0,Plantae,Tracheophyta,Magnoliopsida,Escalloniales,Escalloniaceae,Escallonia,3932735.0
4,Ahaetulla borealis,species,Sauria,genus,https://www.inaturalist.org/observations/82005152,eats,Animalia,Chordata,Squamata,None,Colubridae,Ahaetulla,10893621.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1213197,Hydrocharis morsus-ranae,species,Nymphaea odorata,species,https://www.inaturalist.org/observations/55633954,interactsWith,Plantae,Tracheophyta,Liliopsida,Alismatales,Hydrocharitaceae,Hydrocharis,5329266.0,Plantae,Tracheophyta,Magnoliopsida,Nymphaeales,Nymphaeaceae,Nymphaea,2882412.0
1213198,Hydrocharis morsus-ranae,species,Lemna,genus,https://www.inaturalist.org/observations/55633954,interactsWith,Plantae,Tracheophyta,Liliopsida,Alismatales,Hydrocharitaceae,Hydrocharis,5329266.0,Plantae,Tracheophyta,Liliopsida,Alismatales,Araceae,Lemna,2867567.0
1213199,Hydrocharis morsus-ranae,species,Typha,genus,https://www.inaturalist.org/observations/55633954,interactsWith,Plantae,Tracheophyta,Liliopsida,Alismatales,Hydrocharitaceae,Hydrocharis,5329266.0,Plantae,Tracheophyta,Liliopsida,Poales,Typhaceae,Typha,2702102.0
1213200,Hydrocharis morsus-ranae,species,Ceratophyllum demersum,species,https://www.inaturalist.org/observations/55633954,interactsWith,Plantae,Tracheophyta,Liliopsida,Alismatales,Hydrocharitaceae,Hydrocharis,5329266.0,Plantae,Tracheophyta,Magnoliopsida,Ceratophyllales,Ceratophyllaceae,Ceratophyllum,2882398.0


In [15]:
df.to_parquet("output_file.parquet", index=False)